# 🔮 ORACLE — Phase 1
## Linear Regression & Gradient Descent from First Principles

> *Optimized Regression Analytics for Crypto Live Exchange*

---

This notebook covers the complete theoretical foundation and implementation of linear regression using gradient descent — the statistical backbone of the ORACLE prediction pipeline.

**Contents:**
1. What is Linear Regression?
2. The Residual & The Cost Function (MSE)
3. Why MSE? Squaring vs Absolute Value
4. The MSE Surface — A Convex Bowl
5. Partial Derivatives & The Gradient
6. Gradient Descent & The Learning Rate
7. Deriving the Update Rules from Scratch
8. Implementation
9. Visualization
10. Summary & Limitations

---
## 1. What is Linear Regression?

Linear regression is a technique to **model the relationship between two variables as a straight line**, and use that line to predict unknown values.

Given a dataset of `n` observations `(x₁, y₁), (x₂, y₂), ..., (xₙ, yₙ)`, we want to find a line:

$$\hat{y} = \theta_0 + \theta_1 x$$

where:
- $\hat{y}$ ("y-hat") is the **predicted** value — as opposed to the real `y` from the dataset
- $\theta_0$ is the **intercept** (where the line crosses the y-axis)
- $\theta_1$ is the **slope** (how steeply the line rises or falls)

The goal is to find the values of $\theta_0$ and $\theta_1$ that make $\hat{y}$ as close as possible to the real $y$ across the entire dataset.

---
## 2. The Residual & The Cost Function (MSE)

### The Residual

For each data point, the **residual** is the difference between the real value and the predicted value:

$$e_i = y_i - \hat{y}_i$$

Visually, it is the vertical gap between a data point and the line. A residual of zero means the line passes exactly through that point.

### The Problem with Averaging Raw Residuals

A natural instinct is to average all residuals to measure overall error. But this fails: positive and negative residuals cancel each other out. A line could be catastrophically wrong on every single point and still produce an average residual of zero.

### Mean Squared Error (MSE)

To fix this, we square each residual before averaging — giving us the **Mean Squared Error**:

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

MSE is a single number representing the **average squared distance** between the data points and the line. The smaller it is, the better the line fits the data.

---
## 3. Why MSE? Squaring vs Absolute Value

Two natural ways to make residuals positive exist:

| Approach | Formula | Name |
|---|---|---|
| Absolute value | $\frac{1}{n} \sum |e_i|$ | MAE (Mean Absolute Error) |
| Squaring | $\frac{1}{n} \sum e_i^2$ | MSE (Mean Squared Error) |

The key difference: **squaring amplifies large errors quadratically**.

Consider two models:
- Model A: wrong by 5 units on every prediction
- Model B: wrong by 0 units on most predictions, but wrong by 50 units once

Under MAE, these might look similar. Under MSE, Model B is punished much harder — the error of 50 becomes 2500. This forces the model to especially avoid large mistakes, which is desirable in a trading context where a single catastrophic prediction is far worse than consistent small errors.

Additionally, squaring makes MSE **differentiable everywhere**, which is a mathematical requirement for gradient descent.

---
## 4. The MSE Surface — A Convex Bowl

Substituting $\hat{y}_i = \theta_0 + \theta_1 x_i$ into the MSE formula:

$$MSE(\theta_0, \theta_1) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \theta_0 - \theta_1 x_i)^2$$

MSE is now a function of two parameters: $\theta_0$ and $\theta_1$. If we plot it in 3D (with $\theta_0$ and $\theta_1$ on the horizontal axes and MSE on the vertical axis), we get a **convex bowl shape**.

This is guaranteed because MSE is a sum of squared terms — squares are always non-negative, so the surface can never go below zero, and it has exactly **one global minimum** at the bottom of the bowl.

This is excellent news: there are no local minima to get trapped in. Gradient descent will always find the optimal $\theta_0$ and $\theta_1$, regardless of the starting point.

---
## 5. Partial Derivatives & The Gradient

### Partial Derivative

A **partial derivative** measures how a function changes when one of its inputs changes, while all other inputs are held constant.

For $MSE(\theta_0, \theta_1)$:
- $\frac{\partial MSE}{\partial \theta_0}$ — how MSE changes when $\theta_0$ changes, $\theta_1$ fixed
- $\frac{\partial MSE}{\partial \theta_1}$ — how MSE changes when $\theta_1$ changes, $\theta_0$ fixed

### The Gradient

The **gradient** packages both partial derivatives into a single vector:

$$\nabla MSE = \left(\frac{\partial MSE}{\partial \theta_0}, \frac{\partial MSE}{\partial \theta_1}\right)$$

Geometrically, the gradient points in the direction of **steepest ascent** on the MSE surface. To minimize MSE, we move in the **opposite** direction — hence *gradient descent*.

---
## 6. Gradient Descent & The Learning Rate

Gradient descent is an iterative optimization algorithm. At each step, we move slightly in the direction opposite to the gradient:

$$\theta := \theta - \alpha \cdot \nabla MSE$$

The scalar $\alpha$ (alpha) is the **learning rate** — it controls the size of each step.

| Learning Rate | Consequence |
|---|---|
| Too large | Overshoots the minimum, may diverge |
| Too small | Converges, but extremely slowly |
| Just right | Converges efficiently to the minimum |

Finding a good learning rate is one of the first practical challenges in applying gradient descent. It is typically tuned empirically.

---
## 7. Deriving the Update Rules from Scratch

Starting from:

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \theta_0 - \theta_1 x_i)^2$$

### Partial derivative with respect to $\theta_0$

Using the chain rule: $\frac{d}{d\theta}[f(\theta)]^2 = 2 f(\theta) \cdot f'(\theta)$

The inside function is $f(\theta_0) = y_i - \theta_0 - \theta_1 x_i$, so $f'(\theta_0) = -1$

Since differentiation distributes over summation:

$$\frac{\partial MSE}{\partial \theta_0} = \frac{1}{n} \sum 2(y_i - \theta_0 - \theta_1 x_i) \cdot (-1) = -\frac{2}{n} \sum e_i$$

### Partial derivative with respect to $\theta_1$

Same chain rule, but now $f'(\theta_1) = -x_i$:

$$\frac{\partial MSE}{\partial \theta_1} = -\frac{2}{n} \sum e_i x_i$$

### Update rules

The factor of 2 is absorbed into $\alpha$ (since $\alpha$ is tunable), giving the final update rules:

$$\theta_0 := \theta_0 - \alpha \cdot \left(-\frac{1}{n} \sum e_i\right) = \theta_0 + \frac{\alpha}{n} \sum e_i$$

$$\theta_1 := \theta_1 - \alpha \cdot \left(-\frac{1}{n} \sum e_i x_i\right) = \theta_1 + \frac{\alpha}{n} \sum e_i x_i$$

Both parameters are updated **simultaneously** at each iteration.

---
## 8. Implementation

The following implementation applies gradient descent to find the optimal $\theta_0$ and $\theta_1$ for a given dataset.

In [ ]:
import numpy as np
from typing import Tuple

def linear_regression(x: np.array, y: np.array, alpha: float = 0.1,
                      max_iter: int = 10000, eps: float = 1e-5) -> Tuple:
    """Apply linear regression to (x, y) dataset and compute parameters theta0 and theta1
    of the model with learning rate alpha using gradient descent algorithm.

    Args:
        x (np.array): independent variable
        y (np.array): dependent variable
        alpha (float, optional): learning rate. Defaults to 0.1.
        max_iter (int, optional): maximum number of iterations. Defaults to 10000.
        eps (float, optional): convergence threshold — stops when both gradients
                               fall below this value. Defaults to 1e-5.

    Returns:
        Tuple: (theta0, theta1, i)
            - theta0: y-intercept of the best-fit line
            - theta1: slope of the best-fit line
            - i: number of iterations at which the algorithm stopped
    """
    assert len(x) == len(y), "x and y must have the same length"

    n = len(x)
    # Initialize parameters at origin — safe for linear regression
    # because the MSE surface is convex (one global minimum, no local minima)
    (theta0, theta1) = (0.0, 0.0)

    for i in range(max_iter):
        # Predicted values
        yhat = theta0 + theta1 * x

        # Residuals: how wrong the current line is at each point
        err = y - yhat

        # Partial derivatives of MSE (factor of 2 absorbed into alpha)
        grad_theta0 = (-1 / n) * np.sum(err)
        grad_theta1 = (-1 / n) * np.sum(err * x)

        # Simultaneous parameter update — moving opposite to the gradient
        theta0 = theta0 - alpha * grad_theta0
        theta1 = theta1 - alpha * grad_theta1

        # Convergence check: stop when gradient is flat enough
        if abs(grad_theta0) < eps and abs(grad_theta1) < eps:
            print(f"Converged at iteration {i}")
            break
    else:
        print(f"Warning: reached max_iter={max_iter} without convergence. Consider adjusting alpha or max_iter.")

    return (float(theta0), float(theta1), i)

---
## 9. Visualization

We generate a synthetic dataset with a known linear relationship plus noise, run the regression, and visualize the result.

In [ ]:
import matplotlib.pyplot as plt

# --- Synthetic dataset ---
# True relationship: y = 3 + 2x, with Gaussian noise
np.random.seed(42)
x = np.linspace(0, 10, 100)
y = 3 + 2 * x + np.random.normal(0, 2, size=len(x))

# --- Run regression ---
(theta0, theta1, iters) = linear_regression(x, y, alpha=0.01)
print(f"theta0 (intercept) = {theta0:.4f}  |  expected ≈ 3.0")
print(f"theta1 (slope)     = {theta1:.4f}  |  expected ≈ 2.0")
print(f"Stopped at iteration: {iters}")

# --- Plot ---
yhat = theta0 + theta1 * x

plt.figure(figsize=(9, 5))
plt.scatter(x, y, alpha=0.5, label='Dataset (noisy observations)')
plt.plot(x, yhat, color='red', linewidth=2,
         label=f'Regression line: ŷ = {theta0:.3f} + {theta1:.3f}x')
plt.plot(x, 3 + 2 * x, color='green', linestyle='--', linewidth=1,
         label='True underlying line: y = 3 + 2x')
plt.xlabel('x')
plt.ylabel('y')
plt.title('ORACLE — Phase 1: Linear Regression via Gradient Descent')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---
## 10. Summary & Limitations

### What was built

A linear regression model trained via gradient descent, implemented from scratch using only NumPy. The full pipeline:

```
Dataset → Residuals → MSE → Gradient → Descent → optimal θ₀ and θ₁ → ŷ = θ₀ + θ₁x
```

### Key concepts covered

| Concept | Description |
|---|---|
| Residual | Per-point prediction error: $e_i = y_i - \hat{y}_i$ |
| MSE | Average squared error — the cost function to minimize |
| Convexity | MSE surface is a bowl with one guaranteed global minimum |
| Gradient | Vector of partial derivatives pointing toward steepest ascent |
| Learning rate $\alpha$ | Step size — tradeoff between speed and stability |
| Convergence | Stop when gradient magnitude falls below threshold $\varepsilon$ |

### Known limitations for cryptocurrency prediction

- **Non-stationarity** — Crypto price series are not stationary. Linear regression assumes a stable underlying relationship, which price data violates.
- **Single feature** — This implementation uses one input variable `x`. Real price prediction requires multiple features (lag values, volume, moving averages, etc.).
- **Linearity assumption** — Markets are not linear. This model is a starting point, not a final solution.
- **No temporal awareness** — The model treats data points as independent, ignoring the sequential nature of time series data.

These limitations will be studied empirically in Phase 3 (model evaluation), and are documented here as known challenges to address in future phases.

---

> *"All models are wrong, but some are useful."* — George Box